# Customer Churn Prediction

**Author:** Protik Deb  
**Focus:** Exploratory Data Analysis, classification, model evaluation, and responsible interpretation.

This notebook follows a reproducible workflow on the IBM Telco Customer Churn sample dataset. It compares Logistic Regression and Random Forest using a stratified hold-out set.

> The dataset is a historical sample for a fictional telecom provider. Model metrics are produced when the notebook is executed; no results are assumed in advance.

## 1. Imports and configuration

The workflow uses scikit-learn pipelines so imputation, scaling, and categorical encoding are fitted using training data only.

In [ ]:
from pathlib import Pathimport urllib.requestimport warningsimport numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport seaborn as snsfrom sklearn.compose import ColumnTransformerfrom sklearn.ensemble import RandomForestClassifierfrom sklearn.impute import SimpleImputerfrom sklearn.linear_model import LogisticRegressionfrom sklearn.metrics import (    accuracy_score, precision_score, recall_score, f1_score,    roc_auc_score, classification_report, ConfusionMatrixDisplay,    RocCurveDisplay)from sklearn.model_selection import train_test_splitfrom sklearn.pipeline import Pipelinefrom sklearn.preprocessing import OneHotEncoder, StandardScalerRANDOM_STATE = 42REPO_ROOT = Path.cwd()if REPO_ROOT.name == "notebooks":    REPO_ROOT = REPO_ROOT.parentDATA_DIR = REPO_ROOT / "data"DATA_PATH = DATA_DIR / "Telco-Customer-Churn.csv"DATA_URL = (    "https://raw.githubusercontent.com/IBM/"    "telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv")sns.set_theme(style="whitegrid")warnings.filterwarnings("ignore", category=FutureWarning)

## 2. Load the dataset

The CSV is downloaded from IBM's public sample-data repository if it is not already present locally. The repository is archived, so keep a local copy if the source becomes unavailable.

Source: [IBM Telco Customer Churn sample](https://github.com/IBM/telco-customer-churn-on-icp4d).

In [ ]:
DATA_DIR.mkdir(parents=True, exist_ok=True)if not DATA_PATH.exists():    print("Downloading the IBM Telco Customer Churn sample...")    urllib.request.urlretrieve(DATA_URL, DATA_PATH)df = pd.read_csv(DATA_PATH)print(f"Dataset shape: {df.shape}")display(df.head())

## 3. Initial data quality review

Check column types, missingness, duplicate rows, and the target distribution before modeling.

In [ ]:
display(df.info())display(df.isna().sum().sort_values(ascending=False).to_frame("missing_values").head(10))print("Duplicate rows:", df.duplicated().sum())print("\nTarget counts:")display(df["Churn"].value_counts(dropna=False).to_frame("count"))print("\nTarget proportions:")display((df["Churn"].value_counts(normalize=True) * 100).round(2).to_frame("percent"))

## 4. Cleaning and feature/target separation

- Remove exact duplicate rows.
- Convert `TotalCharges` to numeric; blank strings become missing values.
- Exclude `customerID`, which is an identifier rather than a meaningful predictor.
- Encode the outcome as 1 = churn, 0 = retained.

Missing numeric and categorical predictor values are imputed inside the model pipeline.

In [ ]:
data = df.drop_duplicates().copy()data["TotalCharges"] = pd.to_numeric(data["TotalCharges"], errors="coerce")data["Churn"] = data["Churn"].map({"No": 0, "Yes": 1})if data["Churn"].isna().any():    raise ValueError("Unexpected values in the Churn target column.")target = data["Churn"].astype(int)drop_cols = ["Churn", "customerID"]features = data.drop(columns=[c for c in drop_cols if c in data.columns])print("Rows after duplicate removal:", len(data))print("Missing TotalCharges:", data["TotalCharges"].isna().sum())print("Feature matrix shape:", features.shape)print("Churn rate: {:.1%}".format(target.mean()))

## 5. Exploratory data analysis

These plots describe the sample and should not be interpreted as causal evidence.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))sns.countplot(data=data, x="Churn", ax=axes[0])axes[0].set_title("Customer churn count")axes[0].set_xlabel("Churn (0 = No, 1 = Yes)")(data["Churn"].value_counts(normalize=True).sort_index() * 100).plot(    kind="bar", ax=axes[1])axes[1].set_title("Customer churn share")axes[1].set_xlabel("Churn (0 = No, 1 = Yes)")axes[1].set_ylabel("Percent")axes[1].tick_params(axis="x", rotation=0)plt.tight_layout()plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))sns.boxplot(data=data, x="Churn", y="tenure", ax=axes[0])axes[0].set_title("Tenure by churn outcome")axes[0].set_xlabel("Churn (0 = No, 1 = Yes)")sns.boxplot(data=data, x="Churn", y="MonthlyCharges", ax=axes[1])axes[1].set_title("Monthly charges by churn outcome")axes[1].set_xlabel("Churn (0 = No, 1 = Yes)")plt.tight_layout()plt.show()

In [ ]:
churn_by_contract = (    data.groupby("Contract", observed=False)["Churn"]    .mean()    .sort_values(ascending=False)    .mul(100))display(churn_by_contract.rename("churn_percent").round(1).to_frame())ax = churn_by_contract.plot(kind="bar", figsize=(8, 4))ax.set_title("Observed churn rate by contract type")ax.set_xlabel("Contract")ax.set_ylabel("Churn rate (%)")ax.tick_params(axis="x", rotation=0)plt.tight_layout()plt.show()

## 6. Train/test split and preprocessing

A stratified split keeps the target proportion similar in both subsets. The preprocessing transformer handles numeric and categorical columns separately.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(    features,    target,    test_size=0.20,    random_state=RANDOM_STATE,    stratify=target,)numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()categorical_features = X_train.select_dtypes(exclude=np.number).columns.tolist()numeric_pipeline = Pipeline(steps=[    ("imputer", SimpleImputer(strategy="median")),    ("scaler", StandardScaler()),])categorical_pipeline = Pipeline(steps=[    ("imputer", SimpleImputer(strategy="most_frequent")),    ("onehot", OneHotEncoder(handle_unknown="ignore")),])def make_preprocessor():    """Create a fresh transformer for each model pipeline."""    return ColumnTransformer(        transformers=[            ("numeric", numeric_pipeline, numeric_features),            ("categorical", categorical_pipeline, categorical_features),        ],        remainder="drop",    )print("Training rows:", len(X_train))print("Test rows:", len(X_test))print("Numeric features:", len(numeric_features))print("Categorical features:", len(categorical_features))

## 7. Train and compare models

Both models use the same split and preprocessing. Class weighting is enabled to make the classifiers more attentive to the less frequent churn class. This does not automatically determine the best decision threshold for a business use case.

In [ ]:
models = {    "Logistic Regression": Pipeline(steps=[        ("preprocess", make_preprocessor()),        ("classifier", LogisticRegression(            max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE        )),    ]),    "Random Forest": Pipeline(steps=[        ("preprocess", make_preprocessor()),        ("classifier", RandomForestClassifier(            n_estimators=400,            min_samples_leaf=2,            class_weight="balanced",            random_state=RANDOM_STATE,            n_jobs=-1,        )),    ]),}metric_rows = []fitted_models = {}for name, model in models.items():    model.fit(X_train, y_train)    predictions = model.predict(X_test)    probabilities = model.predict_proba(X_test)[:, 1]    row = {        "Model": name,        "Accuracy": accuracy_score(y_test, predictions),        "Precision": precision_score(y_test, predictions, zero_division=0),        "Recall": recall_score(y_test, predictions, zero_division=0),        "F1": f1_score(y_test, predictions, zero_division=0),        "ROC-AUC": roc_auc_score(y_test, probabilities),    }    metric_rows.append(row)    fitted_models[name] = model    print(f"\n{name}")    print(classification_report(y_test, predictions, zero_division=0))results = pd.DataFrame(metric_rows).set_index("Model").sort_values("ROC-AUC", ascending=False)display(results.round(3))

## 8. Visual model evaluation

Compare ROC curves and confusion matrices. Recall measures the fraction of observed churners detected; precision measures how many predicted churners actually churned in this test sample.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))for name, model in fitted_models.items():    RocCurveDisplay.from_estimator(model, X_test, y_test, name=name, ax=ax)ax.plot([0, 1], [0, 1], linestyle="--", label="Random baseline")ax.set_title("ROC curves on the hold-out test set")plt.tight_layout()plt.show()fig, axes = plt.subplots(1, len(fitted_models), figsize=(11, 4))for ax, (name, model) in zip(axes, fitted_models.items()):    ConfusionMatrixDisplay.from_estimator(        model, X_test, y_test, display_labels=["Retained", "Churn"], ax=ax    )    ax.set_title(name)plt.tight_layout()plt.show()

## 9. Random Forest feature importance

Feature importance is calculated after one-hot encoding. It reflects how this fitted Random Forest used the available features; it is not evidence that a feature causes churn.

In [ ]:
rf_pipeline = fitted_models["Random Forest"]rf_preprocessor = rf_pipeline.named_steps["preprocess"]rf_classifier = rf_pipeline.named_steps["classifier"]feature_names = rf_preprocessor.get_feature_names_out()importance = pd.Series(    rf_classifier.feature_importances_, index=feature_names).sort_values(ascending=False)display(importance.head(15).rename("importance").to_frame().round(4))ax = importance.head(15).sort_values().plot(kind="barh", figsize=(9, 6))ax.set_title("Top 15 Random Forest feature importances")ax.set_xlabel("Importance")ax.set_ylabel("Encoded feature")plt.tight_layout()plt.show()

## 10. Summary and next steps

The output above provides the observed metrics and model diagnostics for this specific random split. Before treating a model as useful, consider:

1. Repeating evaluation with stratified cross-validation and reporting variability.
2. Selecting a probability threshold using realistic contact costs and customer outcomes, not accuracy alone.
3. Checking calibration and subgroup error rates.
4. Testing on newer, independently collected data before any operational use.
5. Reviewing privacy, fairness, and customer-impact implications.

**Interpretation caution:** this is a historical sample dataset for a fictional provider. It does not demonstrate that any intervention will reduce churn, and it is not a production-ready decision system.

In [ ]:
# Select the best model by ROC-AUC for an optional local artifact.best_model_name = results.index[0]best_model = fitted_models[best_model_name]print("Top model by this test split's ROC-AUC:", best_model_name)# Save locally only; model files are excluded from Git by .gitignore.import joblibfrom pathlib import Pathmodel_dir = REPO_ROOT / "models"model_dir.mkdir(exist_ok=True)model_path = model_dir / "best_model.joblib"joblib.dump(best_model, model_path)print("Saved model to:", model_path)

## References

- IBM, [Telco Customer Churn sample dataset repository](https://github.com/IBM/telco-customer-churn-on-icp4d) (archived repository).
- IBM, [Telco Customer Churn sample documentation](https://www.ibm.com/docs/en/cognos-analytics/11.1.0?topic=samples-telco-customer-churn).
- Scikit-learn, [Pipeline and composite estimators](https://scikit-learn.org/stable/modules/compose.html).
- Scikit-learn, [Model evaluation](https://scikit-learn.org/stable/modules/model_evaluation.html).